In [13]:
!pip install transformers
!pip install torchmetrics

In [14]:
import torch
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from torchmetrics import MetricCollection
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassPrecision,
    MulticlassRecall,
    MulticlassF1Score,
    MulticlassAUROC,
    MulticlassAveragePrecision
)

In [15]:
EPOCHS = 10
SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"

In [23]:
df = pd.read_csv("/content/drive/MyDrive/colab ex/datasets/smile-annotations-final.csv", names=["id", "text", "label"])
labels = ['happy', 'angry', 'surprise', 'sad', 'disgust']
df = df[df.label.isin(labels)]

X = df["text"]
y = df["label"]

class2label = {c:idx for idx, c in enumerate(labels)}
y = y.map(class2label)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size = 0.5,
    random_state=SEED
)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class SentimentDataset(Dataset):
  def __init__(self, X, y, tokenizer):
    self.text = X
    self.label = y
    self.tokenizer = tokenizer

  def __len__(self):
    return len(self.label)

  def __getitem__(self, idx):
    text = str(self.text[idx])
    label = self.label[idx]

    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    return {
        'input_ids': encoding['input_ids'].squeeze(0),
        'attention_mask': encoding['attention_mask'].squeeze(0),
        'labels': torch.tensor(label, dtype=torch.long)
    }

train_ds = SentimentDataset(X_train, y_train, tokenizer)
val_ds = SentimentDataset(X_val, y_val, tokenizer)
test_ds = SentimentDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(class2label)).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [25]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)
train_metrics = MetricCollection({
    'acc': MulticlassAccuracy(num_classes=len(class2label), average='micro'),
}).to(device)

val_metrics = MetricCollection({
    'acc':   MulticlassAccuracy(num_classes=len(class2label), average='micro'),
    'prec':  MulticlassPrecision(num_classes=len(class2label), average='macro'),
    'rec':   MulticlassRecall(num_classes=len(class2label), average='macro'),
    'f1':    MulticlassF1Score(num_classes=len(class2label), average='macro'),
    'auroc': MulticlassAUROC(num_classes=len(class2label), average='macro'),
    'pr-auc':    MulticlassAveragePrecision(num_classes=len(class2label), average='macro'),
}).to(device)

In [26]:
def train_epoch(model, loader, loss_fn, optimizer, scheduler, device, metrics):
  model.train()
  metrics.reset()
  total_loss = 0.0
  for batch in loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()
    outputs = model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    loss = loss_fn(logits, labels)

    loss.backward()
    optimizer.step()
    scheduler.step()

    total_loss += loss.item() * len(labels)

    metrics.update(logits, labels)

  return total_loss / len(loader.dataset), metrics.compute()

def eval_epoch(model, loader, device, metrics):
  model.eval()
  metrics.reset()
  with torch.no_grad():
    for batch in loader:
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      outputs = model(input_ids, attention_mask=attention_mask)
      logits = outputs.logits

      metrics.update(logits, labels)

  return metrics.compute()





In [28]:
best_model = None
best_pr = -float("inf")
patience = 3
patience_count = 0
delta = 0.001

for epoch in range(EPOCHS):
  train_loss, train_result = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, device, train_metrics)
  val_result = eval_epoch(model, val_loader, device, val_metrics)

  cur_pr = val_result["pr-auc"]

  print(f"Epoch {epoch:3d} | val " + "  ".join(f"{k}: {v:.4f}" for k,v in val_result.items()))

  if cur_pr > best_pr + delta:
    best_pr = cur_pr
    best_epoch = epoch
    patience_counter = 0
    best_model = model.state_dict()
    print(f"   → New best val pr: {best_pr:.4f} (saved model)")
  else:
    patience_counter += 1
    print(f"   No improvement → patience counter: {patience_counter}/{patience}")

  if patience_counter >= patience:
    print(f"Early stopping triggered at epoch {epoch}")
    break


if best_model:
  model.load_state_dict(best_model)

test_result = eval_epoch(model, test_loader, device, val_metrics)
print("FINAL RESULT")
print(f"Epoch {epoch:3d} | val " + "  ".join(f"{k}: {v:.4f}" for k,v in test_result.items()))

Epoch   0 | val acc: 0.9055  auroc: 0.7779  f1: 0.6740  pr-auc: 0.8636  prec: 0.6528  rec: 0.7716
   → New best val pr: 0.8636 (saved model)
Epoch   1 | val acc: 0.8031  auroc: 0.7404  f1: 0.5133  pr-auc: 0.7570  prec: 0.4635  rec: 0.6400
   No improvement → patience counter: 1/3
Epoch   2 | val acc: 0.9291  auroc: 0.7791  f1: 0.6199  pr-auc: 0.8572  prec: 0.6038  rec: 0.6609
   No improvement → patience counter: 2/3
Epoch   3 | val acc: 0.9370  auroc: 0.7823  f1: 0.6474  pr-auc: 0.8896  prec: 0.6945  rec: 0.6395
   → New best val pr: 0.8896 (saved model)
Epoch   4 | val acc: 0.9528  auroc: 0.7840  f1: 0.6893  pr-auc: 0.8940  prec: 0.7613  rec: 0.6432
   → New best val pr: 0.8940 (saved model)
Epoch   5 | val acc: 0.9528  auroc: 0.7838  f1: 0.6893  pr-auc: 0.8926  prec: 0.7613  rec: 0.6432
   No improvement → patience counter: 1/3
Epoch   6 | val acc: 0.9528  auroc: 0.7836  f1: 0.6893  pr-auc: 0.8926  prec: 0.7613  rec: 0.6432
   No improvement → patience counter: 2/3
Epoch   7 | val a